# 04 · RNN / LSTM factor-model tuning  *(branch `exp/04-rnn-tuning`)*

Brief §3.3. Architectures: **LSTM, GRU, LSTM+GRU hybrid (`enhanced`, the current model), Transformer encoder, TCN**.
Search space: hidden size, depth, sequence length, dropout, learning rate, weight decay, optimiser, batch size,
label smoothing — Optuna TPE + median pruning, one study per architecture, **scored on validation MSE only**.
Every trial is logged (SQLite on Drive + `trials_<arch>.csv`); an interrupted session resumes where it stopped.

Then each architecture's best configuration is retrained with the full budget over `FINAL_SEEDS`, the architecture with
the lowest **mean validation** MSE is selected, and its best-validation seed provides the factors for notebooks 05–07.

**Budget:** worst case `N_TRIALS × TRIAL_MINUTES` per architecture (pruning cuts most trials short) plus the final runs.
Run one architecture per session if needed — nothing is lost.

In [ ]:
# ---- setup: Drive, code, packages (identical in every notebook) -------------------------
BRANCH = "exp/04-rnn-tuning"                                  # the branch this notebook belongs to
REPO = "https://github.com/taimoor-ahmed-1/FinAI-CryptoRaiders.git"
DRIVE_ROOT = "/content/drive/MyDrive/thesis_rl"      # data in, every output out - survives disconnects

import os, sys, subprocess
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/FinAI-CryptoRaiders"
    if not os.path.exists(REPO_DIR):
        # sparse clone: only experiments/ is checked out - the repo's data folders are never downloaded
        subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout", "--branch", BRANCH, REPO, REPO_DIR], check=True)
        subprocess.run(["git", "-C", REPO_DIR, "sparse-checkout", "set", "--no-cone", "/experiments/"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "-B", BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/experiments/requirements-colab.txt"], check=True)
    EXP = f"{REPO_DIR}/experiments"
else:                                                # local run from a checkout: notebooks/ -> experiments/
    EXP = os.path.abspath("..")
    DRIVE_ROOT = os.environ.get("THESIS_RL_ROOT", os.path.abspath("../../thesis_rl_outputs"))
sys.path.insert(0, EXP)
os.chdir(EXP)

import torch
from rltrade import paths, seeding
L = paths.layout(DRIVE_ROOT)
print("code:", BRANCH, seeding.git_commit(EXP), "| device:", "cuda - " + torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available(): print('WARNING: no GPU - Runtime > Change runtime type > T4 GPU')

In [ ]:
seeding.record_environment(L.report / "environment", "04_rnn_tuning")   # exact environment of this run (pip freeze, CUDA, GPU)

In [ ]:
import optuna, json, pandas as pd, matplotlib.pyplot as plt
from rltrade.rnn import tune as T
from rltrade import results as R
optuna.logging.set_verbosity(optuna.logging.WARNING)
ARCHS = ["lstm", "gru", "enhanced", "transformer", "tcn"]
N_TRIALS, TRIAL_STEPS, TRIAL_MINUTES = 20, 3000, 12
FINAL_SEEDS = [0, 1, 2]
FINAL_MAX_MINUTES = 60              # cap per final run (0 = notebook budget + early stopping)

## 1 · Search (validation only)

In [ ]:
studies = {}
for arch in ARCHS:
    print(f"=== {arch}")
    studies[arch] = T.tune_arch(arch, L.prepared, L.rnn_tuning, n_trials=N_TRIALS,
                                max_steps=TRIAL_STEPS, max_minutes=TRIAL_MINUTES)
    s = studies[arch]; print(f"best val MSE {s.best_value:.5f}", s.best_params)
    R.plot_optuna(s, str(L.report / f"rnn_optuna_{arch}"))

## 2 · Full-budget retraining of each best configuration, several seeds

In [ ]:
rows = []
for arch in ARCHS:
    rows += T.final_runs(arch, studies[arch].best_params, L.prepared, L.rnn_tuning, seeds=FINAL_SEEDS,
                         max_minutes=FINAL_MAX_MINUTES)
pd.DataFrame(rows).to_csv(L.rnn_tuning / "final_runs.csv", index=False)
sel, board = T.select(rows, L.rnn_tuning)
board = pd.DataFrame(board); print(board.round(5)); print("SELECTED:", sel)
(L.report / "rnn_leaderboard.md").write_text("# RNN leaderboard (selection on validation MSE)\n\n" + R.df_markdown(board, 5) + f"\n\nSelected: {sel}\n", encoding="utf-8")

## 3 · Loss curves of the final runs

In [ ]:
R.plot_rnn_history({f"{r['arch']} s{r['seed']}": r["dir"] for r in rows if r["seed"] == FINAL_SEEDS[0]},
                   L.report / "rnn_final_loss_curves.png"); plt.show()